# BarqTrain Colab: Benchmark Comparison

This notebook compares three inference setups on two separate benchmark profiles:
- Baseline Hugging Face / PyTorch
- BarqTrain inference patch with CUDA backend enabled
- Unsloth AI with full-weight precision (not 4-bit)

Benchmark profiles:
- `throughput_short`: short prompt + short decode for latency and tokens/sec
- `memory_long`: long prompt + longer decode to stress KV-cache growth

Important Colab flow:
1. Run the install/build cell.
2. Restart the runtime/session.
3. Run the verification/config cell before trusting any benchmark result.


In [ ]:
import os, importlib, importlib.util, shutil, subprocess
from pathlib import Path
from urllib.request import urlopen

GIT_REF = 'codex/phase_24-colab-branch-fix'

if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    script = urlopen('https://sh.rustup.rs').read().decode('utf-8')
    rustup_path = Path('/tmp/rustup-init.sh')
    rustup_path.write_text(script)
    subprocess.run(['sh', str(rustup_path), '-y'], check=True)
os.environ['PATH'] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"
!python -m pip install --upgrade pip setuptools wheel setuptools-rust
!pip install --upgrade uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!uv pip install ninja packaging pandas datasets accelerate peft
if not os.path.isdir('/content/BarqTrain/.git'):
    !git clone -b {GIT_REF} https://github.com/YASSERRMD/BarqTrain.git /content/BarqTrain
else:
    %cd /content/BarqTrain
    !git fetch origin {GIT_REF}
    !git checkout {GIT_REF}
    !git pull --ff-only origin {GIT_REF}
%cd /content/BarqTrain
!git rev-parse --abbrev-ref HEAD
!python -m pip install -e . --no-build-isolation
!BARQTRAIN_BUILD_CUDA=1 python -m pip install -e . --no-build-isolation
importlib.invalidate_caches()
assert shutil.which('cargo'), 'cargo is required for the Rust backend build'
assert shutil.which('ninja'), 'ninja is required for the benchmark'
assert importlib.util.find_spec('barqtrain_rs'), 'barqtrain_rs did not build; do not trust the benchmark'
assert importlib.util.find_spec('barqtrain_cuda'), 'barqtrain_cuda did not build; do not trust the benchmark'
print("Native build finished on branch:", GIT_REF)
print("Restart the runtime/session now, then run the next cell to verify runtime loading.")


In [ ]:
%cd /content/BarqTrain

import gc
import importlib
import importlib.util
import os
import statistics
import sys
import time

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ['BARQTRAIN_AUTO_BUILD'] = '0'
sys.path.insert(0, '/content/BarqTrain/python')
importlib.invalidate_caches()

import barqtrain
import barqtrain._ffi as ffi
assert hasattr(barqtrain, "patch_inference"), "Wrong BarqTrain branch is installed in /content/BarqTrain. Re-run the install cell and make sure it checks out the phase branch."
from barqtrain import patch_inference
from barqtrain.memory import (
    build_generation_kwargs,
    cuda_memory_snapshot,
    generation_overhead_mb as compute_generation_overhead_mb,
)

assert importlib.util.find_spec("barqtrain_rs"), "barqtrain_rs missing after restart"
assert importlib.util.find_spec("barqtrain_cuda"), "barqtrain_cuda missing after restart"
assert ffi.load_rust_backend() is not None, "barqtrain_rs failed to load at runtime"
assert ffi.load_cuda_backend() is not None, "barqtrain_cuda failed to load at runtime"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_MODEL_NAME = 'Qwen/Qwen2-0.5B-Instruct'
UNSLOTH_MODEL_NAME = 'unsloth/Qwen2-0.5B-Instruct'
SHORT_PROMPT = 'Explain why fused kernels can improve LLM training throughput.'
LONG_PROMPT = ('Fused kernels reduce memory traffic and launch overhead. ' * 160).strip()
UNSLOTH_MAX_SEQ_LENGTH = 4096
WARMUP_RUNS = 2
MEASURE_RUNS = 5
BENCHMARK_PROFILES = [
    {"profile": "throughput_short", "prompt": SHORT_PROMPT, "new_tokens": 64, "warmup_runs": 2, "measure_runs": 5, "paged_kv_min_cache_len": 512},
    {"profile": "memory_long", "prompt": LONG_PROMPT, "new_tokens": 256, "warmup_runs": 1, "measure_runs": 3, "paged_kv_min_cache_len": 0},
]

def pick_dtype():
    if DEVICE != 'cuda':
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16

DTYPE = pick_dtype()

print('BarqTrain version:', barqtrain.__version__)
print('barqtrain_rs runtime load:', ffi.load_rust_backend() is not None)
print('barqtrain_cuda runtime load:', ffi.load_cuda_backend() is not None)
print('Device:', DEVICE)
print('Base model:', BASE_MODEL_NAME)
print('Unsloth model:', UNSLOTH_MODEL_NAME)
print('DType:', DTYPE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('This benchmark notebook requires a CUDA runtime. Switch Colab to GPU before running it.')


In [ ]:
def clear_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def load_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_hf_model(model_name, dtype):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        trust_remote_code=True,
    )
    if DEVICE == "cuda":
        model = model.cuda()
    return model


def benchmark_generation(label, model, tokenizer, profile, extra_metrics=None):
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True

    inputs = tokenizer(profile["prompt"], return_tensors="pt")
    prompt_tokens = int(inputs["input_ids"].shape[-1])
    if DEVICE == "cuda":
        inputs = {k: v.cuda() for k, v in inputs.items()}

    clear_memory()
    with torch.inference_mode():
        warmup_kwargs = build_generation_kwargs(model, min(16, profile["new_tokens"]))
        for _ in range(profile["warmup_runs"]):
            _ = model.generate(**inputs, **warmup_kwargs)

    clear_memory()
    resident_snapshot = cuda_memory_snapshot(reset_peak=True)
    setattr(model, "_barqtrain_last_generate_used_paged_kv", False)

    latencies = []
    with torch.inference_mode():
        generate_kwargs = build_generation_kwargs(model, profile["new_tokens"])
        last_token_only = any(generate_kwargs.get(name) == 1 for name in ("logits_to_keep", "num_logits_to_keep"))
        for _ in range(profile["measure_runs"]):
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            outputs = model.generate(**inputs, **generate_kwargs)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            latencies.append(time.perf_counter() - t0)

    peak_snapshot = cuda_memory_snapshot()
    generated_tokens = int(outputs.shape[-1] - inputs["input_ids"].shape[-1])
    avg_latency = sum(latencies) / len(latencies)
    median_latency = statistics.median(latencies)

    result = {
        "profile": profile["profile"],
        "setup": label,
        "dtype": str(DTYPE).replace("torch.", ""),
        "prompt_tokens": prompt_tokens,
        "generated_tokens": generated_tokens,
        "avg_latency_s": avg_latency,
        "median_latency_s": median_latency,
        "tokens_per_sec": generated_tokens / avg_latency if avg_latency > 0 else 0.0,
        "resident_vram_mb": resident_snapshot.allocated_mb,
        "peak_vram_mb": peak_snapshot.max_allocated_mb,
        "generation_overhead_mb": compute_generation_overhead_mb(resident_snapshot, peak_snapshot),
        "paged_kv_cache": bool(getattr(model, "_barqtrain_last_generate_used_paged_kv", False)),
        "last_token_logits_only": last_token_only,
    }
    if extra_metrics:
        result.update(extra_metrics)
    return result


def run_case(label, profile, model_builder, tokenizer_builder, extra_metrics=None, before_run=None):
    clear_memory()
    if before_run is not None:
        before_run(profile)
    tokenizer = tokenizer_builder()
    model = model_builder()
    try:
        return benchmark_generation(label, model, tokenizer, profile, extra_metrics=extra_metrics)
    finally:
        del model
        del tokenizer
        clear_memory()


In [ ]:
results = []

def configure_barqtrain_profile(profile):
    os.environ["BARQTRAIN_ENABLE_PAGED_KV"] = "1"
    os.environ["BARQTRAIN_PAGED_KV_MIN_CACHE_LEN"] = str(profile["paged_kv_min_cache_len"])

for profile in BENCHMARK_PROFILES:
    results.append(
        run_case(
            "baseline_hf_full_weight",
            profile,
            model_builder=lambda: load_hf_model(BASE_MODEL_NAME, DTYPE),
            tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
            extra_metrics={"cuda_backend_loaded": None, "rust_backend_loaded": None},
        )
    )

    results.append(
        run_case(
            "barqtrain_inference_cuda_backend",
            profile,
            model_builder=lambda: patch_inference(load_hf_model(BASE_MODEL_NAME, DTYPE)),
            tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
            extra_metrics={"cuda_backend_loaded": True, "rust_backend_loaded": True},
            before_run=configure_barqtrain_profile,
        )
    )

    try:
        from unsloth import FastLanguageModel

        def build_unsloth_model():
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=UNSLOTH_MODEL_NAME,
                max_seq_length=UNSLOTH_MAX_SEQ_LENGTH,
                dtype=DTYPE,
                load_in_4bit=False,
            )
            FastLanguageModel.for_inference(model)
            if DEVICE == "cuda":
                model = model.cuda()
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            return model, tokenizer

        def unsloth_case():
            clear_memory()
            model, tokenizer = build_unsloth_model()
            try:
                return benchmark_generation(
                    "unsloth_full_weight",
                    model,
                    tokenizer,
                    profile,
                    extra_metrics={"cuda_backend_loaded": None, "rust_backend_loaded": None},
                )
            finally:
                del model
                del tokenizer
                clear_memory()

        results.append(unsloth_case())
    except Exception as e:
        print(f"Unsloth benchmark skipped for {profile['profile']}:", repr(e))


In [ ]:
df = pd.DataFrame(results)
df = df.sort_values(["profile", "tokens_per_sec"], ascending=[True, False]).reset_index(drop=True)
for profile_name in df["profile"].unique():
    print(f"\n=== {profile_name} ===")
    display(df[df["profile"] == profile_name].reset_index(drop=True))
df


## Interpreting the benchmark

This notebook is valid only when the BarqTrain row reports `rust_backend_loaded == True` and `cuda_backend_loaded == True`.

Profile intent:
- `throughput_short`: short decode. Paged KV is expected to stay off because the cache is too small to justify the indirection cost.
- `memory_long`: long decode. Paged KV is expected to turn on and reduce decode-cache growth pressure.

How to read the key columns:
- `profile`: whether the row is the short-throughput or long-memory scenario.
- `prompt_tokens`: prompt length after tokenization.
- `resident_vram_mb`: model + inputs resident on GPU before the measured decode loop.
- `peak_vram_mb`: highest allocated VRAM reached during the measured decode loop.
- `generation_overhead_mb`: decode-time allocation above resident memory.
- `paged_kv_cache`: whether BarqTrain injected its native paged KV cache into `generate()`.
- `last_token_logits_only`: whether generation requested decode-token logits only.

Expected behavior:
- On `throughput_short`, HF or Unsloth may still win. That profile is not where paged KV earns its keep.
- On `memory_long`, the BarqTrain row should be judged primarily on `generation_overhead_mb` and whether `paged_kv_cache == True`.

If BarqTrain still loses on `memory_long` with `paged_kv_cache == True`, that is a real inference-path problem to fix in native code, not a notebook issue.


In [ ]:
out_path = "/content/barqtrain_benchmark_results.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)
